In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

df_h = spark.table("silver.hospital")
df_h.write.format("delta").mode("overwrite").saveAsTable("gold.dim_hospital")
display(spark.table("gold.dim_hospital"))

hospital_id,hospital_name,city,total_beds,load_ts
H002,Fortis Healthcare Delhi,Delhi,600,2026-10-29T00:04:16.838Z
H004,Manipal Hospital,Bangalore,400,2026-10-29T00:04:16.838Z
H005,Yashoda Hospital,Hyderabad,450,2026-10-29T00:04:16.838Z
H001,Apollo Main Hospital,Chennai,850,2026-10-29T00:04:16.838Z
H003,Kokilaben Dhirubhai Hospital,Mumbai,700,2026-10-29T00:04:16.838Z


testing SCD 1 with a small “new load” (H001 changes 850 → 900, and H006 is a new hospital):

In [0]:
from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable

new = spark.createDataFrame(
    [("H001", "Apollo Main Hospital", "Chennai", 900),
     ("H006", "Max Hospital", "Delhi", 500)],
    ["hospital_id", "hospital_name", "city", "total_beds"]
).withColumn("load_ts", current_timestamp())

gold = DeltaTable.forName(spark, "gold.dim_hospital")
(gold.alias("t")
     .merge(new.alias("s"), "t.hospital_id = s.hospital_id")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())

display(spark.table("gold.dim_hospital").orderBy("hospital_id"))

hospital_id,hospital_name,city,total_beds,load_ts
H001,Apollo Main Hospital,Chennai,900,2026-10-29T06:12:39.193Z
H002,Fortis Healthcare Delhi,Delhi,600,2026-10-29T00:04:16.838Z
H003,Kokilaben Dhirubhai Hospital,Mumbai,700,2026-10-29T00:04:16.838Z
H004,Manipal Hospital,Bangalore,400,2026-10-29T00:04:16.838Z
H005,Yashoda Hospital,Hyderabad,450,2026-10-29T00:04:16.838Z
H006,Max Hospital,Delhi,500,2026-10-29T06:12:39.193Z


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

df_p = spark.table("silver.patient")
df_p.write.format("delta").mode("overwrite").saveAsTable("gold.dim_patient")
display(spark.table("gold.dim_patient"))

patient_id,first_name,last_name,gender,date_of_birth,city,load_ts
P004,Sneha,Rao,F,1988-09-19,Hyderabad,2026-10-29T00:16:16.447Z
P003,Kabir,Menon,M,1990-11-02,Bangalore,2026-10-29T00:16:16.447Z
P005,Vikram,Singh,M,1965-05-05,Chennai,2026-10-29T00:16:16.447Z
P001,Aarav,Sharma,M,1980-02-14,Mumbai,2026-10-29T00:16:16.447Z
P002,Riya,Verma,F,1975-07-22,Delhi,2026-10-29T00:16:16.447Z


Step 1: first load

In [0]:
from pyspark.sql.functions import current_timestamp, current_date, lit, col
from delta.tables import DeltaTable

#small new load (P001 moved Mumbai → Pune, P006 is new)
new = spark.createDataFrame(
    [("P001", "Aarav", "Sharma", "M", "1980-02-14", "Pune"),
     ("P006", "Meera", "Nair", "F", "1995-03-10", "Kochi")],
    ["patient_id", "first_name", "last_name", "gender", "date_of_birth", "city"]
).withColumn("date_of_birth", col("date_of_birth").cast("date")) \
 .withColumn("load_ts", current_timestamp()) \
 .withColumn("start_date", current_date()) \
 .withColumn("end_date", lit(None).cast("date")) \
 .withColumn("is_current", lit(True))

df_p = (spark.table("silver.patient")
        .withColumn("start_date", current_date())
        .withColumn("end_date", lit(None).cast("date"))
        .withColumn("is_current", lit(True)))

df_p.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.dim_patient")
display(spark.table("gold.dim_patient"))


patient_id,first_name,last_name,gender,date_of_birth,city,load_ts,start_date,end_date,is_current
P004,Sneha,Rao,F,1988-09-19,Hyderabad,2026-10-29T00:16:16.447Z,2026-10-29,null,true
P003,Kabir,Menon,M,1990-11-02,Bangalore,2026-10-29T00:16:16.447Z,2026-10-29,null,true
P005,Vikram,Singh,M,1965-05-05,Chennai,2026-10-29T00:16:16.447Z,2026-10-29,null,true
P001,Aarav,Sharma,M,1980-02-14,Mumbai,2026-10-29T00:16:16.447Z,2026-10-29,null,true
P002,Riya,Verma,F,1975-07-22,Delhi,2026-10-29T00:16:16.447Z,2026-10-29,null,true


MERGE (close the old row + insert the new patient):

In [0]:
from pyspark.sql.functions import current_timestamp, current_date, lit, col
from delta.tables import DeltaTable
gold = DeltaTable.forName(spark, "gold.dim_patient")
(gold.alias("t")
     .merge(new.alias("s"), "t.patient_id = s.patient_id AND t.is_current = true")
     .whenMatchedUpdate(condition="t.city <> s.city",
                        set={"end_date": "current_date()", "is_current": "false"})
     .whenNotMatchedInsertAll()
     .execute())

display(spark.table("gold.dim_patient").orderBy("patient_id"))

patient_id,first_name,last_name,gender,date_of_birth,city,load_ts,start_date,end_date,is_current
P001,Aarav,Sharma,M,1980-02-14,Mumbai,2026-10-29T00:16:16.447Z,2026-10-29,2026-10-29,false
P002,Riya,Verma,F,1975-07-22,Delhi,2026-10-29T00:16:16.447Z,2026-10-29,null,true
P003,Kabir,Menon,M,1990-11-02,Bangalore,2026-10-29T00:16:16.447Z,2026-10-29,null,true
P004,Sneha,Rao,F,1988-09-19,Hyderabad,2026-10-29T00:16:16.447Z,2026-10-29,null,true
P005,Vikram,Singh,M,1965-05-05,Chennai,2026-10-29T00:16:16.447Z,2026-10-29,null,true
P006,Meera,Nair,F,1995-03-10,Kochi,2026-10-29T06:37:20.660Z,2026-10-29,null,true


In [0]:
new.filter(col("patient_id") == "P001") \
   .write.format("delta").mode("append").saveAsTable("gold.dim_patient")

display(spark.table("gold.dim_patient").orderBy("patient_id", "is_current"))

patient_id,first_name,last_name,gender,date_of_birth,city,load_ts,start_date,end_date,is_current
P001,Aarav,Sharma,M,1980-02-14,Mumbai,2026-10-29T00:16:16.447Z,2026-10-29,2026-10-29,false
P001,Aarav,Sharma,M,1980-02-14,Pune,2026-10-29T06:40:35.092Z,2026-10-29,null,true
P002,Riya,Verma,F,1975-07-22,Delhi,2026-10-29T00:16:16.447Z,2026-10-29,null,true
P003,Kabir,Menon,M,1990-11-02,Bangalore,2026-10-29T00:16:16.447Z,2026-10-29,null,true
P004,Sneha,Rao,F,1988-09-19,Hyderabad,2026-10-29T00:16:16.447Z,2026-10-29,null,true
P005,Vikram,Singh,M,1965-05-05,Chennai,2026-10-29T00:16:16.447Z,2026-10-29,null,true
P006,Meera,Nair,F,1995-03-10,Kochi,2026-10-29T06:37:20.660Z,2026-10-29,null,true
